# Stochastic Calculus, Step by Step · Lesson 6: Convergence and inequalities

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. The law of large numbers · 3. Chebyshev's inequality · 4. Mean to a constant, variance to zero · 5. Preview: squared steps settle, absolute steps blow up · 6. Exercises · 7. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 6,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. The law of large numbers

The average of $n$ independent copies of $X$ settles on $E[X]$, and its standard deviation falls like $\mathrm{sd}(X)/\sqrt n$. Monte Carlo pricing rests on this. Averages of simulated year-end prices $S_T$ against the exact $E[S_T] = S_0e^{rT}$:

In [ ]:
rng = np.random.default_rng(seed)
sim = lambda n: S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * rng.standard_normal(n))
mean_exact = S0 * math.exp(r * T)
sd_exact = math.sqrt(S0**2 * math.exp(2 * r * T) * (math.exp(sigma**2 * T) - 1))
rows = []
for n in (10**2, 10**3, 10**4, 10**5, 10**6):
    x = sim(n)
    rows.append({"n": n, "average": float(x.mean()), "error": float(x.mean() - mean_exact), "sd_of_average": sd_exact / math.sqrt(n)})
# 200 independent averages of 1,000 prices each: how they scatter
avgs = np.array([sim(1000).mean() for _ in range(200)])
out["lln"] = {"mean": mean_exact, "sd": sd_exact, "rows": rows, "avg_sd_1000": float(avgs.std()), "avg_sd_theory": sd_exact / math.sqrt(1000)}
pd.DataFrame(rows)

## 3. Chebyshev's inequality

For any $X$ with finite variance, $P(|X - E[X]| \ge k\,\mathrm{sd}(X)) \le 1/k^2$. It holds for every distribution, so it is loose for any particular one:

In [ ]:
x = sim(2_000_000)
cheb = []
for k in (1.5, 2, 3, 4):
    cheb.append({"k": k, "bound": 1 / k**2, "lognormal": float((np.abs(x - mean_exact) >= k * sd_exact).mean()),
                 "normal": 2 * (1 - N(k))})
# how many paths for |average - mean| < 0.10 with 95% confidence?
eps, alpha = 0.10, 0.05
n_cheb = sd_exact**2 / (alpha * eps**2)
n_clt = (1.959963985 * sd_exact / eps) ** 2
out["cheb"] = {"rows": cheb, "eps": eps, "n_cheb": n_cheb, "n_clt": n_clt, "ratio": n_cheb / n_clt, "var": sd_exact**2}
# Markov for the (non-negative) share price: P(S_T >= 2 E[S_T]) <= 1/2
a_m = 2 * mean_exact
out["markov"] = {"a": a_m, "bound": mean_exact / a_m, "mc": float((x >= a_m).mean()),
                 "exact": 1 - N((math.log(a_m / S0) - (r - 0.5 * sigma**2) * T) / (sigma * math.sqrt(T)))}
pd.DataFrame(cheb)

## 4. Mean to a constant, variance to zero

**The argument used again and again in Part 3.** If $E[X_n] \to c$ and $\mathrm{Var}(X_n) \to 0$, then $X_n \to c$: by Chebyshev, $P(|X_n - E[X_n]| \ge \varepsilon) \le \mathrm{Var}(X_n)/\varepsilon^2 \to 0$. Example: $Q_n = \sum_{i=1}^n Y_i^2$ with $Y_i$ independent $N(0, T/n)$. Then $E[Q_n] = T$ and, using $E[Z^4] = 3$ (Lesson 4), $\mathrm{Var}(Q_n) = 2T^2/n$. So $Q_n \to T$.

In [ ]:
qrows, hists = [], {}
bins = np.linspace(0, 2.5, 101)
for n in (10, 100, 1000, 10_000):
    Y = rng.normal(0, math.sqrt(T / n), (5000, n))
    Q = (Y**2).sum(axis=1)
    A = np.abs(Y).sum(axis=1)
    qrows.append({"n": n, "Q_mean": float(Q.mean()), "Q_sd": float(Q.std()), "Q_sd_theory": math.sqrt(2 / n) * T,
                  "A_mean": float(A.mean()), "A_theory": math.sqrt(2 * n * T / math.pi)})
    if n <= 1000:
        h, _ = np.histogram(Q, bins=bins, density=True)
        hists[str(n)] = h.round(4).tolist()
out["qv"] = {"rows": qrows, "fig": {"mid": ((bins[1:] + bins[:-1]) / 2).round(4).tolist(), "hists": hists,
                                                                        "x_range": [0.0, 2.5],
                                                                        "ticks": [{"x": v, "label": f"{v:g}"} for v in (0, 0.5, 1, 1.5, 2, 2.5)]}}
pd.DataFrame(qrows)

## 5. Preview: squared steps settle, absolute steps blow up

The same steps, added up as absolute values, do not settle: $E\big[\sum|Y_i|\big] = n\sqrt{T/n}\sqrt{2/\pi} = \sqrt{2nT/\pi}$, which grows without limit (the last column above). In Lesson 14 these $Y_i$ become the steps of a Brownian path: its squared steps add up to the elapsed time, while its length is infinite.

## 6. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 1: Solutions*.

**Exercise 1.** Prove Markov's inequality, $P(Y \ge a) \le E[Y]/a$ for $Y \ge 0$, $a > 0$, and deduce Chebyshev's inequality from it.

**Exercise 2.** Let $Y_1, \dots, Y_n$ be independent $N(0, T/n)$ and $Q_n = \sum Y_i^2$. Show $E[Q_n] = T$ and $\mathrm{Var}(Q_n) = 2T^2/n$.

**Exercise 3.** With the same $Y_i$, show $E\big[\sum|Y_i|\big] = \sqrt{2nT/\pi}$.

**Exercise 4.** How many Monte Carlo paths does Chebyshev say you need for $|\bar X_n - E[S_T]| < 0.10$ with probability at least 95%? How many does the normal approximation suggest?

**Exercise 5.** The share price $S_T \ge 0$ has mean $S_0e^{rT} = 103.05$. What does Markov's inequality say about $P(S_T \ge 150)$? Use the lognormal distribution of Lesson 5 to compute the actual probability.

**Exercise 6.** Let $X_n = n$ with probability $1/n$ and $X_n = 0$ otherwise. Show that $X_n \to 0$ in probability, although $E[X_n] = 1$ for every $n$. What happens to $\mathrm{Var}(X_n)$?

**Exercise 7.** (Code) Estimate $P(S_T > K)$ by simulation for $n = 10^3, 10^4, 10^5, 10^6$, with a 95% interval each time, and compare with $N(d_-)$ from Lesson 5.

**Exercise 8.** (Code) Simulate $Q_n$ and $\sum|Y_i|$ for $n$ up to $10^5$ and confirm the formulas of Exercises 6.2 and 6.3.

## 7. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")